In [1]:
import logging.config
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from datetime import datetime,timedelta
from PIL import Image
import re
from tqdm.notebook import tqdm
# from tqdm import tqdm
from typing import List, Dict, Tuple
import csv

import logging
logging.basicConfig(level=logging.WARNING)

In [6]:
import time
for i in tqdm(range(10)):
    time.sleep(.1)
    print(f'{i:05}')

  0%|          | 0/10 [00:00<?, ?it/s]

00000
00001
00002
00003
00004
00005
00006
00007
00008
00009


In [3]:
METEOSAT_ROOT_FOLDER = '/media/vladlanda/DATA/Meteosat'
LI_PROJECTED_FOLDER = 'afa_projected'

DATE_FOLDER_FORMAT = '%Y_%m_%d'

LI_ROOT_FOLDER = f'{METEOSAT_ROOT_FOLDER}{os.sep}{LI_PROJECTED_FOLDER}'

# IR_FOLDER = 'ir_105'
# IR_ROOT_FOLDER = f'{METEOSAT_ROOT_FOLDER}{os.sep}{IR_FOLDER}'

IR_FOLDERS_LIST = ['ir_105','ir_87','ir_97','ir_123']
IR_ROOT_FOLDERS_LIST = [f'{METEOSAT_ROOT_FOLDER}{os.sep}{irf}' for irf in IR_FOLDERS_LIST]

TRAIN_IMAGES_FOLDER = os.path.join('datasets')


In [13]:
def _read_wld(wld_path):
    '''
    https://gdal.org/en/stable/drivers/raster/wld.html
    '''
    wld_keys = ['x_size','rot_y','rot_x','y_size','x_luc','y_luc']
    with open(wld_path, "r") as f:
        wld_dict = {key:float(line.strip()) for key,line in zip(wld_keys,f.readlines())}
    return wld_dict
'''
def _generate_crop_save_images(projected_files,ir_files,wld_files,save_root_folder,lonlat_origin_dict,crop_size = 256):

    IR_REG_EXPRESSION = r'\d{8}T\d{6}Z{1}_\d{8}T\d{6}Z{1}'
    ir_time_reg = re.compile(IR_REG_EXPRESSION)

    for index,(proj_file,ir_file,wld_file) in enumerate(zip(projected_files,ir_files,wld_files)):

        
        file_time = ir_time_reg.findall(ir_file)[0]

        im_proj = plt.imread(proj_file)
        im_proj = np.expand_dims(im_proj,-1)
        im_ir = plt.imread(ir_file)
        im_ir = np.expand_dims(im_ir,-1)
        im_zero = np.zeros_like(im_ir)

        wld_dict = _read_wld(wld_file)

        try:
            im = np.concatenate((im_proj,im_ir,im_zero),axis=-1)
        except: continue

        
        
        for name,lonlat_origin in lonlat_origin_dict.items():

            folder = os.path.join(save_root_folder,name)
            filename = os.path.join(folder,f'{index:04}.bmp')
            if os.path.isfile(filename): continue

            lon,lat = lonlat_origin
            im_lon = wld_dict['x_luc'] - wld_dict['x_size'] / 2 
            im_lat = wld_dict['y_luc'] - wld_dict['y_size'] / 2 

            dlon = lon - im_lon
            dlat = im_lat - lat
            dx_pixels = dlon / wld_dict['x_size']
            dy_pixels = dlat / -wld_dict['y_size']

            dx_pixels = np.floor(dx_pixels).astype(int)
            dy_pixels = np.floor(dy_pixels).astype(int)


            wld_info = f'{wld_dict["x_size"]}\n{wld_dict["rot_y"]}\n{wld_dict["rot_x"]}\n{wld_dict["y_size"]}\n{wld_dict["x_luc"] + wld_dict["x_size"] * dx_pixels}\n{wld_dict["y_luc"] + wld_dict["y_size"] * dy_pixels}\n'

            os.makedirs(folder,exist_ok=True)

            im_to_save = im[dy_pixels:dy_pixels+crop_size,dx_pixels:dx_pixels+crop_size,:]

            # filename = os.path.join(folder,f'{index:04}.jpg')
            filename = os.path.join(folder,f'{index:05}_{file_time}.bmp')

            # plt.imsave(filename,im_to_save)
            pil_im = Image.fromarray(im_to_save, mode='RGB')
            try:
                pil_im.save(filename)
                with open(os.path.join(folder,f'{index:05}_{file_time}.wld'),'w') as f: f.write(wld_info)
            except Exception as e:
                # print(proj_file,ir_file)
                # print(dy_pixels,dy_pixels+crop_size,dx_pixels,dx_pixels+crop_size)
                # print(e)
                pass

    return None
'''

def _get_pairs_of_availble_channels(root_ir :str,root_li: str,additional_channels_folders: List[str],li_source = 'COUNT',upto: datetime = datetime.now()) -> List[Dict]:

    IR_REG_EXPRESSION = r'\d{8}T\d{6}Z{1}_\d{8}T\d{6}Z{1}'
    ir_time_reg = re.compile(IR_REG_EXPRESSION)

    SINGLE_IR_REG_EXPRESSION = r'\d{8}T\d{6}Z{1}'
    single_ir_time_reg = re.compile(SINGLE_IR_REG_EXPRESSION)
    TIME_FORMAT = '%Y%m%dT%H%M%SZ'

    all_root_ir_files = glob.glob(os.path.join(root_ir,'*','*.jpg'))
    all_root_ir_files.sort()
    all_root_li_files = glob.glob(os.path.join(root_li,'*',f'*{li_source}*.jpg'))
    all_root_wld_files = glob.glob(os.path.join(root_ir,'*','*.wld'))
    additional_channels_files = [glob.glob(os.path.join(folder,'*','*.jpg')) for folder in additional_channels_folders]

    list_of_dicts = []

    for main_ir in tqdm(all_root_ir_files):

        # Check if date is less than upto: datetime
        date = single_ir_time_reg.findall(main_ir)[-1]
        ir_date = datetime.strptime(date,TIME_FORMAT)
        if ir_date > upto: continue

        files_dict = {'ir':main_ir}
        time_stamp = ir_time_reg.findall(main_ir)[0]
        try:
            li_file = [li for li in all_root_li_files if time_stamp in li][0]
        except:
            li_file = None
        files_dict['li']=li_file

        for i,channel_files in enumerate(additional_channels_files):
            try:
                ch_file = [ch for ch in channel_files if time_stamp in ch][0]
            except:
                ch_file = None
            files_dict[f'ch{i}']=ch_file

        try:
            wld_file = [wld for wld in all_root_wld_files if time_stamp in wld][0]
        except:
            wld_file = None
        files_dict['wld']=wld_file

        list_of_dicts.append(files_dict)

    return list_of_dicts

def _generate_crop_save_separate_images(source_files: List[Dict],
                                        save_root_folder:str,
                                        lonlat_origin_dict: Dict[str,Tuple[float,float]],
                                        crop_size: int = 256):
        
        # def _check_files_exists(im,wld_dict,type,file_time,save_root_folder,lonlat_origin_dict,crop_size):
        #     all_files_exists = True
        #     for name,lonlat_origin in lonlat_origin_dict.items():
        #         folder = os.path.join(save_root_folder,name)
        #         filename = os.path.join(folder,f'{index:05}_{file_time}_{type}.jpg')
        #         print()
        #     pass

        def _crop_and_save(im,wld_dict,type,file_time,save_root_folder,lonlat_origin_dict,crop_size):

            labels_dict = {}

            for name,lonlat_origin in lonlat_origin_dict.items():
                folder = os.path.join(save_root_folder,name)

                lon,lat = lonlat_origin

                
                im_lon = wld_dict['x_luc'] - wld_dict['x_size'] / 2 
                im_lat = wld_dict['y_luc'] - wld_dict['y_size'] / 2 

                dlon = lon - im_lon
                dlat = im_lat - lat
                dx_pixels = dlon / wld_dict['x_size']
                dy_pixels = dlat / -wld_dict['y_size']

                dx_pixels = np.floor(dx_pixels).astype(int)
                dy_pixels = np.floor(dy_pixels).astype(int)

                wld_info = f'{wld_dict["x_size"]}\n{wld_dict["rot_y"]}\n{wld_dict["rot_x"]}\n{wld_dict["y_size"]}\n{wld_dict["x_luc"] + wld_dict["x_size"] * dx_pixels}\n{wld_dict["y_luc"] + wld_dict["y_size"] * dy_pixels}\n'

                os.makedirs(folder,exist_ok=True)

                im_to_save = im[dy_pixels:dy_pixels+crop_size,dx_pixels:dx_pixels+crop_size]
                filename = os.path.join(folder,f'{index:05}_{file_time}_{type}.jpg')

                # print(labels_dict)

                labels_dict[filename] = np.sum(im_to_save>0)

                plt.imsave(filename,im_to_save,cmap='gray')

                try:
                    wld_save_apth = os.path.join(folder,f'{index:05}_{file_time}.wld')
                    if not os.path.isfile(wld_save_apth):
                        with open(wld_save_apth,'w') as f: 
                            f.write(wld_info)
                except Exception as e:
                    pass

            return labels_dict


        IR_REG_EXPRESSION = r'\d{8}T\d{6}Z{1}_\d{8}T\d{6}Z{1}'
        ir_time_reg = re.compile(IR_REG_EXPRESSION)

        index = 0
        li_labels_dict = {}

        for files_dict in tqdm(source_files):



            wld_file = files_dict['wld']
            try:
                wld_dict = _read_wld(wld_file)
            except: continue
            ir_file = files_dict['ir']
            li_file = files_dict['li']
            file_time = ir_time_reg.findall(ir_file)[0]

            if li_file is None: continue
            # if os.parh.isfile(ir_file):continue

            try:
                ir_im = plt.imread(ir_file)
            except: 
                print(f"Error IR read JPEG : {ir_file}")
                continue
            
            try: 
                _crop_and_save(ir_im,wld_dict,'ir',file_time,save_root_folder,lonlat_origin_dict,crop_size)
            except Exception as e: 
                print(ir_im.shape,ir_file,e); continue
            
            li_im = plt.imread(li_file) 
            li_labels = _crop_and_save(li_im,wld_dict,'li',file_time,save_root_folder,lonlat_origin_dict,crop_size)
            # print(li_labels,li_im)
            li_labels_dict.update(li_labels)

            ch_keys = [key for key in files_dict.keys() if 'ch' in key]
            for ch in ch_keys:
                ch_file = files_dict[ch]
                try: ch_img = plt.imread(ch_file)
                except: ch_img = np.zeros_like(ir_im)
                _crop_and_save(ch_img,wld_dict,ch,file_time,save_root_folder,lonlat_origin_dict,crop_size)
            
            index += 1

        
        with open(os.path.join(save_root_folder,'li_labels.csv'),'w') as csv_file:
            writer = csv.writer(csv_file)
            for key, value in li_labels_dict.items():
                writer.writerow([*tuple(k for k in key.split(os.sep)), value])
        


SIngle folder

In [ ]:
# def get_all_files_triples(li_folder,ir_folder):
# get_all_files_triples(LI_ROOT_FOLDER,IR_ROOT_FOLDER)

li_folder = LI_ROOT_FOLDER
# ir_folder = IR_ROOT_FOLDER

li_source = 'COUNT'
if 'afa' in TRAIN_IMAGES_FOLDER: li_source = 'LI'
all_projected_files = glob.glob(os.path.join(li_folder,'*',f'*{li_source}*.jpg'))
# all_ir_files = [p_file.replace(LI_PROJECTED_FOLDER,IR_FOLDER).replace(f'{li_source}','band') for p_file in all_projected_files]
# all_wld_files = [ir_file.replace('.jpg','.wld') for ir_file in all_ir_files]
# len(all_projected_files),all_projected_files[:3],all_ir_files[:3],all_wld_files[:3]

Multiple Folders

In [ ]:
a = '/media/vladlanda/DATA/Meteosat/ir_105/2024_11_01/FCIL1FDHSI_20241101T000007Z_20241101T000924Z_epct_55270b28_FP_band40.jpg'
IR_REG_EXPRESSION = r'\d{8}T\d{6}Z{1}'
TIME_FORMAT = '%Y%m%dT%H%M%SZ'
ir_time_reg = re.compile(IR_REG_EXPRESSION)

t = ir_time_reg.findall(a)[-1]

datetime.strptime(t,TIME_FORMAT),t


(datetime.datetime(2024, 11, 1, 0, 9, 24), '20241101T000924Z')

In [10]:
root_ir = IR_ROOT_FOLDERS_LIST[0]
root_li = LI_ROOT_FOLDER
additional_channels = IR_ROOT_FOLDERS_LIST[1:]
source_files = _get_pairs_of_availble_channels(root_ir,root_li,additional_channels_folders=additional_channels,upto=datetime(2026,1,1))
source_files[:3]

  0%|          | 0/60554 [00:00<?, ?it/s]

[{'ir': '/media/vladlanda/DATA/Meteosat/ir_105/2024_10_01/FCIL1FDHSI_20241001T000007Z_20241001T000924Z_epct_3f9634a0_FP_band40.jpg',
  'li': '/media/vladlanda/DATA/Meteosat/afa_projected/2024_10_01/FCIL1FDHSI_20241001T000007Z_20241001T000924Z_epct_3f9634a0_FP_COUNT40.jpg',
  'ch0': '/media/vladlanda/DATA/Meteosat/ir_87/2024_10_01/FCIL1FDHSI_20241001T000007Z_20241001T000924Z_epct_79f8eb33_FP_band34.jpg',
  'ch1': '/media/vladlanda/DATA/Meteosat/ir_97/2024_10_01/FCIL1FDHSI_20241001T000007Z_20241001T000924Z_epct_e227550d_FP_band37.jpg',
  'ch2': '/media/vladlanda/DATA/Meteosat/ir_123/2024_10_01/FCIL1FDHSI_20241001T000007Z_20241001T000924Z_epct_f0d3c86d_FP_band43.jpg',
  'wld': '/media/vladlanda/DATA/Meteosat/ir_105/2024_10_01/FCIL1FDHSI_20241001T000007Z_20241001T000924Z_epct_3f9634a0_FP_band40.wld'},
 {'ir': '/media/vladlanda/DATA/Meteosat/ir_105/2024_10_01/FCIL1FDHSI_20241001T001007Z_20241001T001924Z_epct_f54834a0_FP_band40.jpg',
  'li': '/media/vladlanda/DATA/Meteosat/afa_projected/2024

In [14]:

# LONLAT_DICT = {
#                f'Test{os.sep}israel':(  31.348771, 35.164878),
#                # f'Test{os.sep}central_africa_1': ( 27.975000,4.256660),
#                # f'Train{os.sep}central_africa_2':( 20.017528,4.293710),
#                # f'Train{os.sep}central_africa_3':(11.301556,12.091834),
#                f'Train{os.sep}greece':(  20.139946,39.665408),
#                f'Train{os.sep}italy':(10.319818, 41.303870),
#                # f'Train{os.sep}central_africa_4' : (12.029032,4.277996),
#                # f'USA{os.sep}usa_test' : (-59.392079,  8.022814)
#             }


LONLAT_DICT = {
               # f'israel':(  31.348771, 35.164878),
               f'central_africa_1': ( 27.975000,4.256660),
               f'central_africa_2':( 20.017528,4.293710),
               f'central_africa_3':(11.301556,12.091834),
               # f'greece':(  20.139946,39.665408),
               # f'italy':(10.319818, 41.303870),
               f'central_africa_4' : (12.029032,4.277996),
               # f'usa_test' : (-59.392079,  8.022814)
            }

save_root_folder = TRAIN_IMAGES_FOLDER
lonlat_origin_dict = LONLAT_DICT

_generate_crop_save_separate_images(source_files,save_root_folder,lonlat_origin_dict,crop_size=256)

  0%|          | 0/60554 [00:00<?, ?it/s]

(612, 5536) /media/vladlanda/DATA/Meteosat/ir_105/2024_11_25/FCIL1FDHSI_20241125T084007Z_20241125T084023Z_epct_73a334f9_FP_band40.jpg cannot write empty image as JPEG
(2933, 5649) /media/vladlanda/DATA/Meteosat/ir_105/2025_01_21/FCIL1FDHSI_20250121T095007Z_20250121T095525Z_epct_d4ed7e34_FP_band40.jpg cannot write empty image as JPEG
(1016, 5491) /media/vladlanda/DATA/Meteosat/ir_105/2025_04_02/FCIL1FDHSI_20250402T085003Z_20250402T085045Z_epct_685258ca_FP_band40.jpg cannot write empty image as JPEG
Error IR read JPEG : /media/vladlanda/DATA/Meteosat/ir_105/2025_04_24/FCIL1FDHSI_20250424T200007Z_20250424T200935Z_epct_551cd5a1_FP_band40.jpg
Error IR read JPEG : /media/vladlanda/DATA/Meteosat/ir_105/2025_07_18/FCIL1FDHSI_20250718T124007Z_20250718T124934Z_epct_5af03a9a_FP_band40.jpg


#### Split CSV to Train,Test

In [ ]:
# import pandas as pd
# csv_file = os.path.join(save_root_folder,'li_labels.csv')
# # pd_li = pd.read_csv(csv_file,names=['main_folder','datatype_folder','split_type','region','filename','count'])
# pd_li = pd.read_csv(csv_file,names=['main_folder','datatype_folder','region','filename','count'])
# pd_li.insert(loc=0,column='root_folder',value=f'{os.getcwd()}')
# # pd_li = pd.read_csv(csv_file)
# pd_test = pd_li[pd_li['split_type']=='Test']
# pd_train = pd_li[pd_li['split_type']=='Train']


# # Make pos/neg labels equal
# pd_train = pd_train.groupby('count').sample(pd_train.groupby('count').size().min())
# pd_test = pd_test.groupby('count').sample(pd_test.groupby('count').size().min())
# pd_train = pd_train.sample(frac = 1)
# pd_test  = pd_test .sample(frac = 1)


# pd_train.to_csv(os.path.join(save_root_folder,'train_files.csv'),index=False,header=False)
# pd_test.to_csv(os.path.join(save_root_folder,'test_files.csv'),index=False,header=False)

# pd_train,pd_train['count'].sum() / pd_train['count'].count(),pd_test['count'].sum() / pd_test['count'].count()

#### Chronological CSV split

In [ ]:
import pandas as pd
csv_file = os.path.join(save_root_folder,'li_labels.csv')
columns = ['main_folder','datatype_folder','region','filename','count']
pd_li = pd.read_csv(csv_file,names=columns)
pd_li.insert(loc=0,column='root_folder',value=f'{os.getcwd()}')
pd_li

In [ ]:
import pandas as pd
csv_file = os.path.join(save_root_folder,'li_labels.csv')
columns = ['main_folder','datatype_folder','region','filename','count']
pd_li = pd.read_csv(csv_file,names=columns)
pd_li.insert(loc=0,column='root_folder',value=f'{os.getcwd()}')
pd_li

# Apply string to date
IR_REG_EXPRESSION = r'\d{8}T\d{6}Z{1}'
DATETIME_FORMAT = '%Y%m%dT%H%M%SZ'
time_reg = re.compile(IR_REG_EXPRESSION)
print(len(pd_li))
pd_li['date'] = pd_li.apply(lambda row: datetime.strptime(time_reg.findall(row.filename)[0],DATETIME_FORMAT),axis=1)

# Sort by date
pd_li = pd_li.sort_values(['date'],ignore_index=True)

# Split to Train/Test
n_items = len(pd_li)
p_train = .75
n_train = int(p_train * n_items)
n_test  = n_items - n_train

assert n_items == n_train+n_test
# pd_test = pd_li[pd_li['split_type']=='Test']
pd_train = pd_li[:n_train]
pd_test  = pd_li[n_train:]

# Get pathces where atleast 2.5% of pixels is lightning
n_pixels = 256.0**2
li_ratio = .005
pd_train = pd_train[pd_train['count'] / n_pixels > li_ratio]

regions = ['central_africa_1','central_africa_2','central_africa_3','central_africa_4']

# Train Israel,greece, Italy region filter
# mask = (pd_train['region']=='israel')# | (pd_train['region']=='greece') | (pd_train['region']=='italy')
mask = pd_train['region'].isin(regions)
pd_train = pd_train[mask]

# Test Israel region filter
# pd_test = pd_test[pd_test['region']=='israel']

# Equalize pos2neg to 50%
# pd_li = pd_li.groupby('count').sample(pd_li.groupby('count').size().min())
# po2neg_ratio = pd_li['count'].sum() / pd_li['count'].count()

# Select only israel for test and equalize Train/Test ratio to 50%
# pd_test = pd_test[pd_test['region'] == 'israel']

pd_test = pd_test[pd_test['region'].isin(regions)]
pd_test = pd_test[pd_test['count'] / n_pixels > li_ratio]
# pd_test = pd_test.groupby('count').sample(pd_test.groupby('count').size().min())

pd_train[columns].to_csv(os.path.join(save_root_folder,'train_files_chrono_africa.csv'),index=False,header=False)
pd_test[columns].to_csv(os.path.join(save_root_folder,'test_files_chrono_africa.csv'),index=False,header=False)

n_items,pd_train['count'].sum() / pd_train['count'].count(),pd_test['count'].sum() / pd_test['count'].count()

In [ ]:
pd_train

In [ ]:
pd_test

#### GEOTiffs and Plotting

In [ ]:
from rasterio.transform import from_origin
import rasterio

for name in LONLAT_DICT.keys():

    folder = os.path.join(TRAIN_IMAGES_FOLDER,name)

    wld_data = list(_read_wld(os.path.join(folder,'0002.wld')).values())
    im = plt.imread(os.path.join(folder,'0002.bmp'))

    pixel_x = wld_data[0]  # Pixel size in X direction (degrees per pixel)
    pixel_y = wld_data[3]  # Negative Pixel size in Y direction
    upper_left_x = wld_data[4]  # Top-left X coordinate (Longitude)
    upper_left_y = wld_data[5]  # Top-left Y coordinate (Latitude)

    # Define raster transformation
    transform = from_origin(upper_left_x, upper_left_y, pixel_x, -pixel_y)
    output_tif = f"test_{os.path.basename(folder)}.tif"
    with rasterio.open(
        output_tif,
        "w",
        driver="GTiff",
        height=im.shape[1],
        width=im.shape[0],
        count=3,
        dtype=im.dtype,
        crs="EPSG:4326",  # Geographic coordinate system
        transform=transform
    ) as dst:
        dst.write(np.moveaxis(im,-1,0))

    print(f"Aligned product saved as {output_tif}")

In [ ]:

%matplotlib inline
plt.title(os.path.basename(all_ir_files[im_index]))
plt.imshow(im_ir,cmap='gray')
plt.imshow(im_proj,alpha=0.5)

# plt.imshow(np.concatenate((im_proj,im_ir,np.ones_like(im_ir)*0),axis=-1))

plt.colorbar()
plt.show()

In [ ]:
mu,sig = 0,1.2
a = np.random.randn(1000) * sig + mu
b = np.random.normal(-1.2,sig,1000)
# b = np.exp(a)
plt.hist(np.exp(a),bins=100,density=True)
plt.hist(np.exp(b),bins=100,density=True)
plt.show()

In [ ]:
np.random.seed(2)
i = np.random.randn(25).reshape(5,5)
i = i > 0
plt.imshow(i)
plt.colorbar()
plt.show()
np.sum(i > 0)